# HCl rovibrational spectrum

*physchem · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/hcl_rovibrational)

The infrared band of HCl is a comb of lines: the molecule vibrates (an anharmonic oscillator) and rotates (a rotor that stretches as it spins), and every line is a jump $v = 0 \to 1$ with $\Delta J = \pm 1$. The P and R branches of both chlorine isotopes come out as one batched $(I, J)$ grid, with Boltzmann intensities for a temperature ramp; a combination-differences least-squares fit then recovers $B_e$, $\alpha_e$ and $\nu_0$ from the lines.

**Vector form**

$$G(v) = \omega_e(v+\tfrac12) - \omega_e x_e(v+\tfrac12)^2,\qquad F_v(J) = B_vJ(J+1) - D_eJ^2(J+1)^2,\qquad B_v = B_e - \alpha_e(v+\tfrac12),\qquad \tilde\nu = G(1) - G(0) + F_1(J^\prime) - F_0(J^{\prime\prime})$$

**Index (tensor) form**

$$F_{viJ} = B_{vi}\,J(J+1) - D_i\,J^2(J+1)^2 \qquad \tilde\nu^{R,P}_{iJ} = \nu_{0i} + F_{1i,J\pm1} - F_{0iJ} \qquad I_{tiJ} = a_i\,(2J+1)\,e^{-hcF_{0iJ}/kT_t}/Q_{ti} \qquad S_{tg} = I_{t\ell}\,g_{\ell g} \qquad \beta_{isk} = X^{+}_{kn}\,y_{isn}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\omega_{e,i},\ B_{e,i},\ \ldots$ | `constants(mu)` | (I,) | approximate H³⁵Cl constants, scaled to H³⁷Cl with $\rho = \sqrt{\mu_{35}/\mu_{37}}$ |
| $F_{viJ}$ | `einsum('i,j->ij', B, J*(J+1)) - einsum('i,j->ij', D, (J*(J+1))**2)` | (I, J) | rotational terms of level $v$; no index summed, outer products |
| $\tilde\nu^{R}_{iJ},\ \tilde\nu^{P}_{iJ}$ | `branches(J, c)` | (I, J), (I, J) | every line of both isotopologues in one grid; $J$ is the lower level |
| $p_{tiJ}$ | `populations(J, B0, D, T)` | (T, I, J) | Boltzmann population of level $J$ at each temperature of the ramp |
| $I^{R,P}_{tiJ}$ | `einsum('i,tij->tij', a, p) * (J+1 or J) / (2J+1)` | (T, I, J) | abundance × population × Hönl–London factor; P(0) gets exactly zero |
| $S_{tg} = I_{t\ell}\,g_{\ell g}$ | `einsum('tl,lg->tg', I, g)` | (T, G) | repeated $\ell$ ⇒ summed over all lines: the broadened spectrum |
| $y_{isn}$ | `combination_differences(R, P) / (4J+2)` | (I, 2, n) | $\Delta_2F/(4J+2) = B_v - 2D(J^2+J+1)$ for the lower ($s=0$) and upper ($s=1$) state |
| $\beta_{isk} = X^{+}_{kn}y_{isn}$ | `einsum('kn,isn->isk', pinv(X), y)` | (I, 2, 2) | repeated $n$ ⇒ summed: least-squares $B_v$ and $D$ for every isotopologue and state |

The constants are approximate literature values for H³⁵Cl: $\omega_e \approx 2991$, $\omega_ex_e \approx 52.8$, $B_e \approx 10.59$, $\alpha_e \approx 0.31$ and $D_e \approx 5\times10^{-4}$ cm⁻¹. The band origin $\nu_0 = \omega_e - 2\omega_ex_e \approx 2885$ cm⁻¹ carries no line, because a $\Sigma$ state like HCl's has no $\Delta J = 0$ (Q) branch: the gap between R(0) and P(1) is about $4B$. Because $B_1 < B_0$ (the vibrating bond is longer on average), the R lines crowd together towards high $J$ while the P lines spread apart. Each line is a doublet, H³⁷Cl sitting about 2 cm⁻¹ lower with a third of the intensity (natural abundance 76 : 24); the shift grows with $|\nu - \nu_0|$ because every constant scales with a power of the reduced mass. As the gas warms, the Boltzmann factor $(2J+1)e^{-hcF(J)/kT}$ moves its maximum to higher $J$ and the two branches spread out (each frame is scaled to its strongest line). Combination differences are the spectroscopist's trick for separating the two states: R(J−1) and P(J+1) end on the same upper level, so their difference depends only on $v = 0$; R(J) and P(J) start from the same lower level, so theirs depends only on $v = 1$. A straight-line fit of $\Delta_2F/(4J+2)$ against $J^2 + J + 1$ gives $B_v$ (the intercept) and $D$ (the slope), then $\alpha_e = B_0 - B_1$ and $B_e = B_0 + \alpha_e/2$. Add measurement noise to see how well the constants survive. The equivalent Morse well depth is $\omega_e^2/4\omega_ex_e$. Simplifications: no Herman–Wallis intensity corrections, one Lorentzian width for every line, and absorption not weighted by $\tilde\nu$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/hcl_rovibrational/main.py
"""Main loop: constants → P and R lines → Boltzmann intensities and spectra for a temperature ramp → fit → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

N_FRAMES = 60


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    names = tensors.isotope_names()
    mu = tensors.reduced_masses(dev)  # (I,) u
    c = tensors.constants(mu)  # dict of (I,) cm⁻¹
    J = tensors.rotational_levels(params["J_max"], dev)  # (J,)

    R, P = equation.branches(J, c)  # (I, J) cm⁻¹ each
    B0 = equation.rot_constant(0, c["Be"], c["ae"])
    T = tensors.temperatures(params["T"], N_FRAMES, dev)  # (F,) K, the last one is T
    pop = equation.populations(J, B0, c["De"], T)  # (F, I, J)
    I_R, I_P = equation.intensities(J, pop, tensors.abundances(dev))  # (F, I, J) each

    step = min(0.1, params["fwhm"] / 8)
    grid = tensors.wavenumber_grid(P[:, 1:].min().item() - 25, R.max().item() + 25, step, dev)  # (G,)
    S = solve.spectra(R, P, I_R, I_P, grid, params["fwhm"])  # (F, G)

    # "measure" the lines (with a little noise) and fit the constants back by combination differences
    gen = torch.Generator(device="cpu").manual_seed(0)
    noise = lambda x: params["noise"] * torch.randn(x.shape, generator=gen).to(dev)  # noqa: E731
    fit = solve.fit_constants(R + noise(R), P + noise(P), J)

    nu0 = equation.band_origin(c["we"], c["wexe"])
    print(f"T = {params['T']:.0f} K   ·   J ≤ {params['J_max']}   ·   line width {params['fwhm']:g} cm⁻¹")
    for i, name in enumerate(names):
        print(
            f"{name}: μ = {mu[i].item():.4f} u   ν₀ = {nu0[i].item():.2f}   B₀ = {B0[i].item():.3f}   "
            f"B₁ = {equation.rot_constant(1, c['Be'], c['ae'])[i].item():.3f} cm⁻¹   "
            f"Morse depth {equation.morse_depth(c['we'], c['wexe'])[i].item():.0f} cm⁻¹"
        )
    print("lines of H³⁵Cl (cm⁻¹):  " + "  ".join(f"R({j})={R[0, j].item():.1f}" for j in range(4)))
    print("                        " + "  ".join(f"P({j})={P[0, j].item():.1f}" for j in range(1, 5)))
    print(
        f"band gap R(0) − P(1) = {(R[0, 0] - P[0, 1]).item():.1f} cm⁻¹ ≈ 2(B₀ + B₁), "
        f"no line at ν₀ (ΔJ = 0 is forbidden)"
    )
    print(
        f"R spacing shrinks {(R[0, 1] - R[0, 0]).item():.2f} → {(R[0, -1] - R[0, -2]).item():.2f}, "
        f"P spacing grows {(P[0, 1] - P[0, 2]).item():.2f} → {(P[0, -2] - P[0, -1]).item():.2f} cm⁻¹  (B₁ < B₀)"
    )
    iso = R[0] - R[1]
    j8 = min(8, len(J) - 1)
    print(
        f"isotope splitting ν(H³⁵Cl) − ν(H³⁷Cl): ν₀ {(nu0[0] - nu0[1]).item():.2f}, "
        f"R(0) {iso[0].item():.2f}, R({j8}) {iso[j8].item():.2f} cm⁻¹"
    )
    J_peak = int(pop[-1, 0].argmax())
    J_estimate = (params["T"] / (2 * equation.HC_K * B0[0].item())) ** 0.5 - 0.5
    print(f"most populated level at {params['T']:.0f} K: J = {J_peak}  (≈ √(kT/2hcB) − ½ = {J_estimate:.1f})")
    print(f"fit (H³⁵Cl, noise {params['noise']:g} cm⁻¹):        true      fitted")
    for key, label in (("Be", "B_e"), ("ae", "α_e"), ("nu0", "ν₀")):
        true = (nu0 if key == "nu0" else c[key])[0].item()
        print(f"   {label:<4} {true:12.4f} {fit[key][0].item():12.4f}")
    print(f"   D    {c['De'][0].item():12.2e} {fit['D'][0].item():12.2e}")

    return plot.animate(grid, S, T, R, P, I_R, J, pop, fit, nu0, names, out_dir / "hcl_rovibrational.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/hcl_rovibrational/equation.py
r"""Physics → tensor translation.

A diatomic molecule vibrates (anharmonic oscillator) and rotates (a rotor that stretches a little when it spins).
In wavenumbers (cm⁻¹) its term values are

    G(v)   = ω_e (v + ½) − ω_e x_e (v + ½)²
    F_v(J) = B_v J(J+1) − D_e J²(J+1)²,        B_v = B_e − α_e (v + ½)

The fundamental band v = 0 → 1 obeys ΔJ = ±1 (no Q branch for HCl), so for isotopologue i and lower level J

    R branch (J → J+1):   ν^R_{iJ} = ν₀_i + F_{1,i}(J+1) − F_{0,i}(J)
    P branch (J → J−1):   ν^P_{iJ} = ν₀_i + F_{1,i}(J−1) − F_{0,i}(J),      ν₀ = G(1) − G(0) = ω_e − 2ω_e x_e

Line strengths follow the Boltzmann population of the lower level times the Hönl–London factor:

    I^R_{iJ} = a_i (J+1) e^{−hc F_{0,i}(J)/kT} / Q_i,       I^P_{iJ} = a_i J e^{−hc F_{0,i}(J)/kT} / Q_i

and the spectrum is the sum of a line shape over every line ℓ = (i, branch, J):

    S_{tg} = Σ_ℓ I_{tℓ} g_{ℓg}            (ℓ summed; t = temperature frame, g = wavenumber grid point)
"""

import torch

HC_K = 1.438777  # second radiation constant hc/k in cm K


def vib_term(v: float, we: torch.Tensor, wexe: torch.Tensor) -> torch.Tensor:
    """G(v) = ω_e(v+½) − ω_e x_e(v+½)²   (I,) cm⁻¹."""
    return we * (v + 0.5) - wexe * (v + 0.5) ** 2


def band_origin(we: torch.Tensor, wexe: torch.Tensor) -> torch.Tensor:
    """ν₀ = G(1) − G(0) = ω_e − 2 ω_e x_e   (I,)."""
    return vib_term(1, we, wexe) - vib_term(0, we, wexe)


def rot_constant(v: float, Be: torch.Tensor, ae: torch.Tensor) -> torch.Tensor:
    """B_v = B_e − α_e(v + ½): a vibrating bond is longer on average, so it rotates more slowly   (I,)."""
    return Be - ae * (v + 0.5)


def rot_term(J: torch.Tensor, B: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    """F_iJ = B_i J(J+1) − D_i J²(J+1)².   J: (J,), B, D: (I,) → (I, J).   No index summed: outer products."""
    x = J * (J + 1)
    return torch.einsum("i,j->ij", B, x) - torch.einsum("i,j->ij", D, x**2)


def branches(J: torch.Tensor, c: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
    """Line positions ν^R_{iJ}, ν^P_{iJ}   (I, J) cm⁻¹, for every isotopologue i and lower level J at once.

    P(0) does not exist (J = −1); its position is meaningless but its strength below is exactly zero.
    """
    nu0 = band_origin(c["we"], c["wexe"])[:, None]  # (I, 1)
    B0, B1 = rot_constant(0, c["Be"], c["ae"]), rot_constant(1, c["Be"], c["ae"])
    F0 = rot_term(J, B0, c["De"])  # lower level, v = 0
    R = nu0 + rot_term(J + 1, B1, c["De"]) - F0
    P = nu0 + rot_term(J - 1, B1, c["De"]) - F0
    return R, P


def populations(J: torch.Tensor, B0: torch.Tensor, D: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    """Boltzmann fraction of molecules in level J: p_tiJ = (2J+1) e^{−hcF₀(J)/kT_t} / Q_ti   (T, I, J)."""
    F0 = rot_term(J, B0, D)  # (I, J)
    w = (2 * J + 1) * torch.exp(-HC_K * F0[None] / T[:, None, None])
    return w / w.sum(-1, keepdim=True)  # Q = Σ_J (2J+1) e^{−…}: the rotational partition function


def intensities(J: torch.Tensor, p: torch.Tensor, abundance: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """I^R_tiJ = a_i p_tiJ (J+1)/(2J+1),   I^P_tiJ = a_i p_tiJ J/(2J+1)   (T, I, J) each."""
    base = torch.einsum("i,tij->tij", abundance, p) / (2 * J + 1)
    return base * (J + 1), base * J


def lorentzian(nu_lines: torch.Tensor, grid: torch.Tensor, fwhm: float) -> torch.Tensor:
    """g_ℓg = (γ/π) / ((ν̃_g − ν̃_ℓ)² + γ²), γ = FWHM/2: the shape of one line   (L, G)."""
    gamma = 0.5 * fwhm
    return (gamma / torch.pi) / ((grid[None, :] - nu_lines[:, None]) ** 2 + gamma**2)


def spectrum(I_lines: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """S_tg = Σ_ℓ I_tℓ g_ℓg   (T, G).   Repeated ℓ ⇒ summed over every line."""
    return torch.einsum("tl,lg->tg", I_lines, g)


def combination_differences(R: torch.Tensor, P: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Differences that share an upper (or a lower) level, so the other state cancels.  J = 1 … J_max − 1.

    Δ₂F″(J) = R(J−1) − P(J+1) = F₀(J+1) − F₀(J−1)    (same upper level J: depends only on v = 0)
    Δ₂F′(J) = R(J)   − P(J)   = F₁(J+1) − F₁(J−1)    (same lower level J: depends only on v = 1)
    Both equal (4J+2) [B_v − 2 D (J² + J + 1)].       R, P: (I, J) → (I, J−2) each.
    """
    lower = R[:, :-2] - P[:, 2:]
    upper = R[:, 1:-1] - P[:, 1:-1]
    return lower, upper


def morse_depth(we: torch.Tensor, wexe: torch.Tensor) -> torch.Tensor:
    """Well depth of the equivalent Morse potential, D_e = ω_e² / (4 ω_e x_e)   (cm⁻¹)."""
    return we**2 / (4 * wexe)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/hcl_rovibrational/tensors.py
"""Build the tensors: spectroscopic constants of each isotopologue, the J grid, the wavenumber grid, temperatures.

Units: wavenumbers in cm⁻¹, temperatures in K.
"""

import torch

# Approximate spectroscopic constants of H³⁵Cl (ground electronic state), rounded literature values.
HCL35 = {
    "we": 2990.9,  # ω_e, harmonic wavenumber
    "wexe": 52.8,  # ω_e x_e, anharmonicity
    "Be": 10.593,  # B_e, equilibrium rotational constant
    "ae": 0.307,  # α_e, vibration–rotation coupling
    "De": 5.3e-4,  # D_e, centrifugal distortion
}
# atomic masses (u) and natural abundances
M_H = 1.007825
ISOTOPES = [("H³⁵Cl", 34.968853, 0.758), ("H³⁷Cl", 36.965903, 0.242)]


def isotope_names() -> list[str]:
    return [name for name, _, _ in ISOTOPES]


def reduced_masses(device: torch.device) -> torch.Tensor:
    """μ_i = m_H m_Cl / (m_H + m_Cl)   (I,) in u."""
    m_cl = torch.tensor([m for _, m, _ in ISOTOPES], device=device)
    return M_H * m_cl / (M_H + m_cl)


def abundances(device: torch.device) -> torch.Tensor:
    """Natural abundance of each isotopologue   (I,)."""
    return torch.tensor([a for _, _, a in ISOTOPES], device=device)


def constants(mu: torch.Tensor) -> dict[str, torch.Tensor]:
    """Constants of every isotopologue (each (I,)), scaled from H³⁵Cl with ρ = √(μ₃₅/μ_i).

    Same potential curve, different mass: ω_e ∝ ρ, ω_e x_e ∝ ρ², B_e ∝ ρ², α_e ∝ ρ³, D_e ∝ ρ⁴.
    """
    rho = torch.sqrt(mu[0] / mu)
    powers = {"we": 1, "wexe": 2, "Be": 2, "ae": 3, "De": 4}
    return {key: HCL35[key] * rho ** powers[key] for key in powers}


def rotational_levels(J_max: int, device: torch.device) -> torch.Tensor:
    """J = 0, 1, …, J_max   (J,): the rotational quantum number of the lower (v = 0) level."""
    return torch.arange(J_max + 1, device=device, dtype=torch.get_default_dtype())


def wavenumber_grid(lo: float, hi: float, step: float, device: torch.device) -> torch.Tensor:
    """ν̃ grid of the simulated spectrum   (G,) cm⁻¹."""
    return torch.arange(lo, hi, step, device=device)


def temperatures(T_end: float, n_frames: int, device: torch.device) -> torch.Tensor:
    """A temperature ramp from 40 K up to T_end   (F,): one spectrum per animation frame."""
    return torch.linspace(40.0, max(T_end, 41.0), n_frames, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/hcl_rovibrational/solve.py
"""Build the spectrum for every temperature, and fit B_e, α_e, D and ν₀ back from the line positions."""

import torch

from .equation import combination_differences, lorentzian, rot_term, spectrum


def spectra(R, P, I_R, I_P, grid: torch.Tensor, fwhm: float) -> torch.Tensor:
    """Spectrum S (T, G) from lines R, P (I, J) with strengths I_R, I_P (T, I, J).

    All isotopologues, both branches and every J are flattened into one line index ℓ, then summed in one einsum.
    """
    nu_lines = torch.cat([R.flatten(), P.flatten()])  # (L,)
    I_lines = torch.cat([I_R.flatten(1), I_P.flatten(1)], dim=1)  # (T, L)
    return spectrum(I_lines, lorentzian(nu_lines, grid, fwhm))


def fit_constants(R: torch.Tensor, P: torch.Tensor, J: torch.Tensor) -> dict[str, torch.Tensor]:
    """Least-squares fit of the measured lines R, P (I, J) by combination differences.

    y_isn = Δ₂F_s(J_n) / (4J_n + 2) = B_s − 2D (J_n² + J_n + 1)   for s = v″ = 0 (lower) and v′ = 1 (upper)
    is a straight line in x_n = J_n² + J_n + 1, so one design matrix X_nk = [1, −2x_n] serves every isotopologue
    and both states:  β_isk = X⁺_kn y_isn   (n summed; X⁺ is the pseudo-inverse = the least-squares solution).
    """
    Jm = J[1:-1]  # J = 1 … J_max − 1
    lower, upper = combination_differences(R, P)  # (I, n) each
    y = torch.stack([lower, upper], dim=1) / (4 * Jm + 2)  # (I, 2, n)
    X = torch.stack([torch.ones_like(Jm), -2 * (Jm**2 + Jm + 1)], dim=1)  # (n, 2)
    beta = torch.einsum("kn,isn->isk", torch.linalg.pinv(X), y)  # (I, 2, 2): [B_v, D] for v = 0, 1
    B0, B1, D = beta[:, 0, 0], beta[:, 1, 0], beta[:, :, 1].mean(1)
    alpha = B0 - B1
    Be = B0 + 0.5 * alpha
    # with B₀, B₁, D known, every line gives ν₀ = ν − F₁(J′) + F₀(J″); average over R(J) and P(J ≥ 1)
    F0 = rot_term(J, B0, D)
    nu0_R = R - rot_term(J + 1, B1, D) + F0
    nu0_P = (P - rot_term(J - 1, B1, D) + F0)[:, 1:]
    nu0 = torch.cat([nu0_R, nu0_P], dim=1).mean(1)
    return {"B0": B0, "B1": B1, "D": D, "ae": alpha, "Be": Be, "nu0": nu0, "x": Jm**2 + Jm + 1, "y": y}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/hcl_rovibrational/plot.py
"""Animate the HCl band as the gas warms up: spectrum, an isotope doublet, rotational populations and the fit."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif


def animate(grid, S, T, R, P, I_R, J, pop, fit, nu0, names, path: Path) -> Path:
    g, S, T = grid.cpu().numpy(), S.cpu().numpy(), T.cpu().numpy()
    R, P, J, pop, nu0 = R.cpu().numpy(), P.cpu().numpy(), J.cpu().numpy(), pop.cpu().numpy(), nu0.cpu().numpy()
    S = S / S.max(1, keepdims=True)  # each frame scaled to its strongest line, so the envelope shape is easy to read

    fig = plt.figure(figsize=(8.4, 5.0))
    grid_spec = fig.add_gridspec(2, 3, height_ratios=[1.25, 1])
    ax_s = fig.add_subplot(grid_spec[0, :])
    ax_z = fig.add_subplot(grid_spec[1, 0])
    ax_f = fig.add_subplot(grid_spec[1, 1])
    ax_p = fig.add_subplot(grid_spec[1, 2])

    # --- the whole fundamental band ---
    ax_s.set_xlim(g[0], g[-1])
    ax_s.set_ylim(0, 1.25)
    ax_s.set_xlabel("wavenumber (cm⁻¹)", labelpad=1)
    ax_s.set_ylabel("absorption (arb.)")
    ax_s.axvline(nu0[0], color="#9ca3af", ls="--", lw=1)
    ax_s.text(nu0[0], 1.13, "ν₀: band gap, no Q branch", ha="center", fontsize=8, color="#6b7280")
    ax_s.text(0.5 * (nu0[0] + g[-1]), 1.05, "R branch  (ΔJ = +1)", ha="center", fontsize=9, color="#2563eb")
    ax_s.text(0.5 * (nu0[0] + g[0]), 1.05, "P branch  (ΔJ = −1)", ha="center", fontsize=9, color="#2563eb")
    for label, x in (("R(0)", R[0, 0]), ("P(1)", P[0, 1])):
        ax_s.text(
            x, -0.02, label, ha="center", va="top", fontsize=7, color="#dc2626", transform=ax_s.get_xaxis_transform()
        )
    (line_s,) = ax_s.plot([], [], color="#2563eb", lw=0.9)
    title = ax_s.set_title("HCl fundamental band")

    # --- zoom on one line: the H³⁵Cl / H³⁷Cl doublet, 3 : 1 by natural abundance ---
    j = min(3, len(J) - 2)
    centre = 0.5 * (R[0, j] + R[1, j])
    window = (g > centre - 5) & (g < centre + 5)
    ax_z.set_xlim(centre - 5, centre + 5)
    ax_z.set_ylim(0, S[:, window].max() * 1.25)
    ax_z.set_xlabel("cm⁻¹", labelpad=1)
    ax_z.set_title(f"R({j}) isotope doublet", fontsize=10)
    ax_z.set_yticks([])
    for i, name in enumerate(names):
        ax_z.axvline(R[i, j], color="#9ca3af", ls=":", lw=1)
        ax_z.text(R[i, j], 0.92, name, transform=ax_z.get_xaxis_transform(), ha="center", fontsize=7.5)
    (line_z,) = ax_z.plot([], [], color="#2563eb", lw=1.4)

    # --- combination-difference fit (static: line positions do not depend on T) ---
    x, y = fit["x"].cpu().numpy(), fit["y"][0].cpu().numpy()  # H³⁵Cl
    xs = np.linspace(0, x.max() * 1.05, 50)
    D = fit["D"][0].item()
    for s, (label, key, color) in enumerate((("v = 0", "B0", "#2563eb"), ("v = 1", "B1", "#dc2626"))):
        B = fit[key][0].item()
        ax_f.plot(x, y[s], "o", ms=3, color=color)
        ax_f.plot(xs, B - 2 * D * xs, color=color, lw=1, label=f"{label}: B = {B:.3f}")
    ax_f.set_xlabel("J² + J + 1", labelpad=1)
    ax_f.set_ylabel(r"$\Delta_2F\,/\,(4J+2)$  (cm⁻¹)", fontsize=8)
    ax_f.set_title("combination differences", fontsize=10)
    ax_f.legend(fontsize=7, loc="lower left")
    ax_f.text(
        0.97, 0.95, f"α_e = {fit['ae'][0].item():.3f}\nB_e = {fit['Be'][0].item():.3f}",
        transform=ax_f.transAxes, ha="right", va="top", fontsize=7.5,
    )  # fmt: skip

    # --- Boltzmann population of the rotational levels of v = 0 ---
    bars = ax_p.bar(J, pop[0, 0], color="#93c5fd", edgecolor="#1e40af", lw=0.6)
    ax_p.set_xlim(-0.7, J[-1] + 0.7)
    ax_p.set_ylim(0, pop[:, 0].max() * 1.08)
    ax_p.set_xlabel("J (v = 0)", labelpad=1)
    ax_p.set_title("level populations", fontsize=10)

    def update(step: int):
        line_s.set_data(g, S[step])
        line_z.set_data(g[window], S[step, window])
        for bar, h in zip(bars, pop[step, 0]):
            bar.set_height(h)
        title.set_text(f"HCl fundamental band  v = 0 → 1   ·   T = {T[step]:.0f} K")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(T), len(T)), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T': 300.0,
    'J_max': 15,
    'fwhm': 0.6,
    'noise': 0.02
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.hcl_rovibrational")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.hcl_rovibrational.main")
problem_dir = Path("problems/physchem/hcl_rovibrational")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
